# Project 1: Final Model (One Model per Setting)

The final model picks, **for each of the 9 settings** $(n, k, m)$, the best of the trained networks, or the average of several. Each test sample is then sent to the model for its setting. The project explicitly allows this ("a separate DNN for each specific tuple $(n, k, m)$, … like a mixture-of-experts model").

**Candidates for a setting:**
- that setting's V5 specialist networks (`dnn_v5_{k}-{m}`, plus `dnn_v5_{k}-{m}_ft200` where a longer fine-tuning run exists);
- the all-setting ensemble (`ensemble`: V2 + V4 run A with AdamW + both V3 transformers).

Each candidate alone, and the average of any two or three candidates, is scored on that setting's provided-data **validation** rows. The lowest validation cost is chosen; held-out test costs are only reported. All candidates share the same seeded split, so no candidate was trained on these rows.

Averages are built as one Keras model with an `Average` layer over the members' $\log_2$ predictions, so every per-setting model consists only of neural-network layers. Its output stays within `[0, 100]`, so every prediction $2^z$ is finite and at least 1.


In [ ]:
# Install dependencies (Google Colab only; local/NERSC runs use the project environment).
# The versions are pinned to those used for training, so the saved model loads and
# predicts identically.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numpy>=2,<3" "tensorflow==2.21.0" "keras==3.15.1"


## Configuration

| Setting | Default | Environment variable |
|---|---|---|
| All-setting candidates (comma-separated folders in `artifacts/`) | `ensemble` | `MHEIGHT_GENERAL` |
| Largest number of models averaged for one setting | 3 | `MHEIGHT_MAX_MEMBERS` |
| Output directory | `artifacts/final` | `MHEIGHT_RUN` |


In [ ]:
import itertools
import json
import os
import pickle
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 636
GENERAL = [run for run in os.environ.get("MHEIGHT_GENERAL", "ensemble").split(",") if run]
MAX_MEMBERS = int(os.environ.get("MHEIGHT_MAX_MEMBERS", 3))
ARTIFACT_DIR = Path("artifacts") / os.environ.get("MHEIGHT_RUN", "final")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PARAMETER_SETTINGS = [(9, k, m) for k in (4, 5, 6) for m in range(2, 9 - k + 1)]
EVAL_DEVICE = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"

tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__, "| evaluation device:", EVAL_DEVICE)
CANDIDATES = {}
for n, k, m in PARAMETER_SETTINGS:
    specialists = [run for run in (f"dnn_v5_{k}-{m}", f"dnn_v5_{k}-{m}_ft200") if (Path("artifacts") / run / "best_model.keras").exists()]
    CANDIDATES[(n, k, m)] = specialists + GENERAL
    print((n, k, m), "candidates:", CANDIDATES[(n, k, m)])


## Load the Provided Data and the Shared Split

Only the provided data is needed. The split is read from a candidate's `split_indices.npz`; indices below `N_PROVIDED` refer to provided rows. The check confirms that all candidates used the same validation and test rows.


In [ ]:
NUMPY_MAJOR = int(np.__version__.split(".")[0])


class NumpyCompatUnpickler(pickle.Unpickler):
    # NumPy 2 pickles reference numpy._core; NumPy 1.x only provides numpy.core.
    def find_class(self, module, name):
        if NUMPY_MAJOR < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, "rb") as file:
        return NumpyCompatUnpickler(file).load()


def pack_inputs(samples):
    """All three member inputs: padded P, padded G = [I_k | P], and [n, k, m]."""
    matrices = np.zeros((len(samples), 6, 5), dtype=np.float32)
    generators = np.zeros((len(samples), 6, 9), dtype=np.float32)
    parameters = np.zeros((len(samples), 3), dtype=np.float32)
    for row, (n, k, m, P) in enumerate(samples):
        P = np.asarray(P, dtype=np.float64)
        if P.shape != (k, n - k):
            raise ValueError(f"Sample {row}: P has shape {P.shape}, expected {(k, n - k)}")
        matrices[row, :k, :n - k] = P
        generators[row, :k, :k] = np.eye(k)
        generators[row, :k, k:n] = P
        parameters[row] = (n, k, m)
    return {"matrix": matrices, "generator": generators, "parameters": parameters}


features = load_pickle("data/CSCE-636-Project-1-Train-n_k_m_P")
labels = np.asarray(load_pickle("data/CSCE-636-Project-1-Train-mHeights"), dtype=np.float64)
N_PROVIDED = len(features)
splits = {}
for run in sorted({run for runs in CANDIDATES.values() for run in runs}):
    split_file = Path("artifacts") / run / "split_indices.npz"
    if not split_file.exists():
        continue  # e.g. the ensemble, whose members were checked when it was built
    saved = np.load(split_file)
    current = {name: saved[name][saved[name] < N_PROVIDED] for name in ("validation", "test")}
    if splits and any(not np.array_equal(current[name], splits[name]) for name in current):
        raise ValueError(f"{run} used a different split")
    splits = current
inputs = pack_inputs(features)
setting_ids = np.array([PARAMETER_SETTINGS.index((n, k, m)) for n, k, m, _ in features], dtype=np.int8)
log2_labels = np.log2(labels)
print(f"Provided rows: {N_PROVIDED:,}; validation {len(splits['validation']):,}; test {len(splits['test']):,}")


## Score Every Candidate and Combination per Setting

Each candidate predicts its setting's validation and test rows (on the GPU when available). For every setting, all single candidates and all averages of up to `MAX_MEMBERS` candidates are scored by $\sigma_{n,k,m}$ on the validation rows; the lowest wins.


In [ ]:
def load_member(run, device=EVAL_DEVICE):
    with tf.device(device):
        return tf.keras.models.load_model(str(Path("artifacts") / run / "best_model.keras"), compile=False)


def member_inputs(network, batch):
    return {tensor.name: batch[tensor.name] for tensor in network.inputs}


def predict_log2(network, indices, device=EVAL_DEVICE, batch_size=8192):
    @tf.function(jit_compile=True)
    def forward(batch):
        return network(batch, training=False)

    out = np.empty(len(indices), dtype=np.float64)
    with tf.device(device):
        for start in range(0, len(indices), batch_size):
            selected = indices[start:start + batch_size]
            batch = member_inputs(network, {name: values[selected] for name, values in inputs.items()})
            out[start:start + len(selected)] = np.asarray(forward(batch)).reshape(-1)
    return out


def setting_costs(indices, predicted):
    errors = (log2_labels[indices] - predicted) ** 2
    return np.array([errors[setting_ids[indices] == g].mean() for g in range(len(PARAMETER_SETTINGS))])


def rows_of(split, setting):
    idx = splits[split]
    return idx[setting_ids[idx] == PARAMETER_SETTINGS.index(setting)]


def sigma(indices, predicted):
    return float(((log2_labels[indices] - predicted) ** 2).mean())


choices, report = {}, {}
loaded = {}
for setting, runs in CANDIDATES.items():
    preds = {}
    for run in runs:
        if run not in loaded:
            loaded[run] = load_member(run)
        preds[run] = {split: predict_log2(loaded[run], rows_of(split, setting)) for split in ("validation", "test")}
    options = []
    for size in range(1, min(MAX_MEMBERS, len(runs)) + 1):
        for combo in itertools.combinations(runs, size):
            val = sigma(rows_of("validation", setting), np.mean([preds[r]["validation"] for r in combo], axis=0))
            test = sigma(rows_of("test", setting), np.mean([preds[r]["test"] for r in combo], axis=0))
            options.append((val, size, combo, test))
    options.sort()
    val, _, combo, test = options[0]
    choices[setting] = list(combo)
    report[str(setting)] = {"members": list(combo), "validation_sigma": val, "test_sigma": test,
                            "singles": {r: {"validation": sigma(rows_of("validation", setting), preds[r]["validation"]),
                                            "test": sigma(rows_of("test", setting), preds[r]["test"])} for r in runs}}
    singles = "  ".join(f"{r}: {v['validation']:.4f}" for r, v in report[str(setting)]["singles"].items())
    print(f"{setting}: chosen {' + '.join(combo)} | validation {val:.4f} | test {test:.4f}   (single validation: {singles})")
print(f"\nMean over settings: validation {np.mean([r['validation_sigma'] for r in report.values()]):.4f} | "
      f"test {np.mean([r['test_sigma'] for r in report.values()]):.4f}")


## Build and Save One Model per Setting

Each setting's model takes the same three inputs (`matrix`, `generator`, `parameters`) and averages its members' outputs with an `Average` layer (a single member is wrapped the same way). It is saved as `final_k{k}_m{m}.keras`; reloading reproduces the same predictions.


In [ ]:
def build_ensemble(runs):
    model_inputs = {
        "matrix": tf.keras.Input(shape=(6, 5), name="matrix"),
        "generator": tf.keras.Input(shape=(6, 9), name="generator"),
        "parameters": tf.keras.Input(shape=(3,), name="parameters"),
    }
    outputs = []
    for index, run in enumerate(runs):
        member = load_member(run, "/CPU:0")
        member.name = f"member_{index}"
        outputs.append(member({tensor.name: model_inputs[tensor.name] for tensor in member.inputs}))
    output = tf.keras.layers.Average(name="log2_height")(outputs) if len(outputs) > 1 else tf.keras.layers.Identity(name="log2_height")(outputs[0])
    return tf.keras.Model(inputs=model_inputs, outputs=output, name="mheight_ensemble")


for (n, k, m), members in choices.items():
    path = ARTIFACT_DIR / f"final_k{k}_m{m}.keras"
    model = build_ensemble(members)
    model.save(path)
    reloaded = tf.keras.models.load_model(str(path), compile=False)
    check = {name: values[rows_of("test", (n, k, m))[:2048]] for name, values in inputs.items()}
    if not np.array_equal(np.asarray(model(check, training=False)), np.asarray(reloaded(check, training=False))):
        raise RuntimeError(f"The reloaded model for {(n, k, m)} does not reproduce the saved model")
    print(f"Saved {path} ({reloaded.count_params():,} parameters): {members}")
(ARTIFACT_DIR / "choices.json").write_text(json.dumps(report, indent=2) + "\n")


## Test the Final Model

The σ check routes each provided-data test row to its setting's saved model, **on the CPU with fixed batches of 1024** (the final cell's path), saves the predictions as a pickle list, reads them back and computes $\sigma_{n,k,m}$ per setting.


In [ ]:
cpu_log2 = np.empty(len(splits["test"]))
position = {row: i for i, row in enumerate(splits["test"])}
for (n, k, m) in PARAMETER_SETTINGS:
    rows = rows_of("test", (n, k, m))
    with tf.device("/CPU:0"):
        network = tf.keras.models.load_model(str(ARTIFACT_DIR / f"final_k{k}_m{m}.keras"), compile=False)
    cpu_log2[[position[r] for r in rows]] = predict_log2(network, rows, device="/CPU:0", batch_size=1024)
prediction_path = ARTIFACT_DIR / "local_test_predictions.pkl"
with prediction_path.open("wb") as file:
    pickle.dump(np.exp2(cpu_log2).tolist(), file)
with prediction_path.open("rb") as file:
    reread = np.log2(np.asarray(pickle.load(file)))
final_costs = {}
for setting in PARAMETER_SETTINGS:
    mask = setting_ids[splits["test"]] == PARAMETER_SETTINGS.index(setting)
    final_costs[str(setting)] = float(((log2_labels[splits["test"]][mask] - reread[mask]) ** 2).mean())
    print(f"sigma{setting} = {final_costs[str(setting)]:.6f}")
print(f"CPU path (as in the final cell), test: mean over settings {np.mean(list(final_costs.values())):.6f}")
(ARTIFACT_DIR / "costs.json").write_text(json.dumps({"test_cpu": final_costs, "choices": report}, indent=2) + "\n")


## Final Cell: Predict m-Heights for a Test File

This is the cell for testing the submitted model. It is **self-contained**: in a fresh session, run the first cell (it installs the pinned TensorFlow/Keras versions on Colab), then run this cell. None of the training cells or the training data are needed.

1. Place the 9 per-setting models next to this notebook as `final_k{k}_m{m}.keras` (for example `final_k6_m2.keras`). When run from the project folder, the cell also finds them in `artifacts/final/`.
2. Set `TEST_INPUT_PATH` to the test file: a pickled list of samples `[n, k, m, P]`, in the same format as `CSCE-636-Project-1-Train-n_k_m_P`.
3. Run the cell. It writes `OUTPUT_PATH`: a pickled list of predicted m-heights (Python floats), one per input sample in the same order, in the same format as `CSCE-636-Project-1-Train-mHeights`.

Each sample goes to the model for its $(k, m)$. `P` is placed in a zero-padded `6 × 5` array, $G = [I_k \mid P]$ in a zero-padded `6 × 9` array, and `[n, k, m]` is passed alongside them. Each model outputs $z = \log_2 \hat y$ in `[0, 100]`, and the only post-processing is $\hat y = 2^z$. Every prediction is therefore finite and at least 1; the cell checks this before saving. Inference runs on the CPU in fixed batches of 1024, compiled with XLA, so repeated runs on a machine give identical results.


In [ ]:
# ===== Final cell: load the trained DNN and predict m-heights for a test file =====
import pickle
from pathlib import Path

import numpy as np
import tensorflow as tf

TEST_INPUT_PATH = "CSCE-636-Project-1-Test-n_k_m_P"   # pickled list of [n, k, m, P]
OUTPUT_PATH = "CSCE-636-Project-1-Test-mHeights"      # pickled list of predicted m-heights
BATCH = 1024


class CompatUnpickler(pickle.Unpickler):
    # Pickles written with NumPy 2 reference numpy._core; NumPy 1.x only has numpy.core.
    def find_class(self, module, name):
        if int(np.__version__.split(".")[0]) < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


if not Path(TEST_INPUT_PATH).exists():
    # Nothing to predict yet (e.g. a batch run before the test file is available).
    print(f"Test file {TEST_INPUT_PATH!r} not found: set TEST_INPUT_PATH and run this cell again.")
else:
    def specialist_file(k, m):
        for path in (f"final_k{k}_m{m}.keras", f"artifacts/final/final_k{k}_m{m}.keras"):
            if Path(path).exists():
                return path
        raise FileNotFoundError(f"No model found for k={k}, m={m}")

    with open(TEST_INPUT_PATH, "rb") as file:
        samples = CompatUnpickler(file).load()

    # Inputs: P padded to 6 x 5, G = [I_k | P] padded to 6 x 9, and [n, k, m].
    matrices = np.zeros((len(samples), 6, 5), dtype=np.float32)
    generators = np.zeros((len(samples), 6, 9), dtype=np.float32)
    parameters = np.zeros((len(samples), 3), dtype=np.float32)
    for row, (n, k, m, P) in enumerate(samples):
        P = np.asarray(P, dtype=np.float64)
        if P.shape != (k, n - k):
            raise ValueError(f"Sample {row}: P has shape {P.shape}, expected {(k, n - k)}")
        matrices[row, :k, :n - k] = P
        generators[row, :k, :k] = np.eye(k)
        generators[row, :k, k:n] = P
        parameters[row] = (n, k, m)

    # Each sample goes to the model for its (k, m); outputs are z = log2(m-height).
    log2_heights = np.full(len(samples), np.nan)
    used = []
    for k, m in sorted({(int(p[1]), int(p[2])) for p in parameters}):
        rows = np.flatnonzero((parameters[:, 1] == k) & (parameters[:, 2] == m))
        model_file = specialist_file(k, m)
        used.append(model_file)
        with tf.device("/CPU:0"):
            network = tf.keras.models.load_model(model_file, compile=False)

        @tf.function(jit_compile=True)
        def predict_log2(batch):
            return network(batch, training=False)

        with tf.device("/CPU:0"):
            for start in range(0, len(rows), BATCH):
                selected = rows[start:start + BATCH]
                batch = {"matrix": matrices[selected], "generator": generators[selected], "parameters": parameters[selected]}
                log2_heights[selected] = np.asarray(predict_log2(batch)).reshape(-1)
    predicted_heights = np.exp2(log2_heights)

    if len(predicted_heights) != len(samples) or not np.isfinite(predicted_heights).all() \
            or np.any(predicted_heights < 1):
        raise ValueError("Every prediction must be finite and at least 1; nothing was saved")
    with open(OUTPUT_PATH, "wb") as file:
        pickle.dump(predicted_heights.tolist(), file)

    print(f"Models used: {used} (TensorFlow {tf.__version__}, Keras {tf.keras.__version__})")
    print(f"Predicted {len(predicted_heights):,} m-heights -> {OUTPUT_PATH}")
    print(f"Range: {predicted_heights.min():.4f} to {predicted_heights.max():.4f}; first five: {predicted_heights[:5].tolist()}")
